## Basic experimentation

In [1]:
%pip install "torch==2.4.1"
%pip install "transformers==4.57.6"
%pip install "torch-directml==0.2.5.dev240914"

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
import transformers

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PyTorch available:", transformers.utils.is_torch_available())

d:\Programs\Anaconda\envs\llm_test\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.4.1+cpu
Transformers: 4.57.6
PyTorch available: True


In [3]:
import torch_directml

# Check if DirectML can see your AMD GPU
if torch_directml.is_available():
    print("🚀 Success! DirectML is available and ready to use.")
    print(f"Device Name: {torch_directml.device_name(0)}") # Prints your exact AMD GPU model
else:
    print("❌ DirectML is not available. Check your installation or graphics drivers.")

🚀 Success! DirectML is available and ready to use.
Device Name: AMD Radeon(TM) Graphics 


In [4]:
from transformers import AutoModel, AutoTokenizer

model_name = "google-bert/bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

model.eval()

print("Model loaded!")
print("Parameters:", sum(p.numel() for p in model.parameters()))
print("Device:", next(model.parameters()).device)

Model loaded!
Parameters: 109482240
Device: cpu


In [5]:
text = "Machine learning is fascinating."
inputs = tokenizer(text, return_tensors="pt")

with torch.inference_mode():
    outputs = model(**inputs)

print("Token IDs:", inputs["input_ids"])
print("Hidden-state shape:", outputs.last_hidden_state.shape)

Token IDs: tensor([[  101,  3698,  4083,  2003, 17160,  1012,   102]])
Hidden-state shape: torch.Size([1, 7, 768])


In [6]:
from transformers import AutoModelForMaskedLM

mlm_model = AutoModelForMaskedLM.from_pretrained(model_name)

text = "Conspiracy [MASK]."
inputs = tokenizer(text, return_tensors="pt")

with torch.inference_mode():
    outputs = mlm_model(**inputs)

mask_position = (inputs["input_ids"] == tokenizer.mask_token_id).nonzero()[0, 1]
logits = outputs.logits[0, mask_position]
top_ids = logits.topk(5).indices

print(tokenizer.convert_ids_to_tokens(top_ids.tolist()))

Some weights of the model checkpoint at google-bert/bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


['theory', 'theories', 'theorist', 'theorists', 'magazine']


In [7]:
from huggingface_hub import scan_cache_dir

cache = scan_cache_dir()

for repo in cache.repos:
    print(repo.repo_id, repo.size_on_disk)

Qwen/Qwen2.5-0.5B-Instruct 999586347
google-bert/bert-base-uncased 441147956


In [8]:
sentences = [
    "She sat on the river bank.",
    "She deposited money at the bank."
]

for text in sentences:
    inputs = tokenizer(text, return_tensors="pt")

    with torch.inference_mode():
        outputs = model(**inputs)

    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    bank_position = tokens.index("bank")
    vector = outputs.last_hidden_state[0, bank_position]

    print(f"\nSentence: {text}")
    print("Token:", tokens[bank_position])
    print("Vector shape:", vector.shape)
    print("First 10 dimensions:", vector[:10])


Sentence: She sat on the river bank.
Token: bank
Vector shape: torch.Size([768])
First 10 dimensions: tensor([-0.0262, -0.5946, -0.1989, -0.0932, -0.6516,  0.1346,  0.4127,  1.3314,
        -0.1530, -0.6545])

Sentence: She deposited money at the bank.
Token: bank
Vector shape: torch.Size([768])
First 10 dimensions: tensor([ 0.3933, -0.4198, -0.2935,  0.0195,  0.8532,  0.1560, -0.2703,  0.8467,
         0.1838, -0.2685])


In [9]:
tokens

['[CLS]', 'she', 'deposited', 'money', 'at', 'the', 'bank', '.', '[SEP]']

In [10]:
import torch.nn.functional as F

In [11]:
vectors = []

sentences = [
    "She sat on the river bank.",
    "She deposited money at the bank."
]

for text in sentences:
    inputs = tokenizer(text, return_tensors="pt")

    with torch.inference_mode():
        outputs = model(**inputs)

    #print(outputs.last_hidden_state.shape)
    
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    bank_position = tokens.index("bank")

    vectors.append(outputs.last_hidden_state[0, bank_position])

similarity = F.cosine_similarity(
    vectors[0].unsqueeze(0),
    vectors[1].unsqueeze(0)
)

print("Cosine similarity:", similarity.item())

Cosine similarity: 0.5157915949821472


In [12]:
text = "She is playing outside."

inputs = tokenizer(text, return_tensors="pt")
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

print(tokens)

['[CLS]', 'she', 'is', 'playing', 'outside', '.', '[SEP]']


In [13]:
print(inputs)

{'input_ids': tensor([[ 101, 2016, 2003, 2652, 2648, 1012,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1]])}


In [14]:
def get_word_embedding(text, word, tokenizer, model):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        return_offsets_mapping=True
    )

    # Locate the word in the original text.
    start = text.index(word)
    end = start + len(word)

    # Find tokens that overlap the word's character span.
    offsets = inputs["offset_mapping"][0].tolist()
    positions = [
        i for i, (s, e) in enumerate(offsets)
        if s < end and e > start
    ]

    # Run BERT (offset_mapping is tokenizer metadata, not a model input).
    model_inputs = {
        k: v for k, v in inputs.items()
        if k != "offset_mapping"
    }

    with torch.inference_mode():
        outputs = model(**model_inputs)

    # Average the vectors if the word spans multiple tokens.
    return outputs.last_hidden_state[0, positions].mean(dim=0)

In [15]:
sentences = [
    "Where is Tom?",
    "Computer mouse."
]

for i, text in enumerate(sentences):
    inputs = tokenizer(text, return_tensors="pt")
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    print(f"Sentence {i}: {text}")
    print(list(enumerate(tokens)))
    print()

token_indices = [1, 2]  # Select a token index for each sentence

vectors = []

for text, index in zip(sentences, token_indices):
    inputs = tokenizer(text, return_tensors="pt")

    with torch.inference_mode():
        outputs = model(**inputs)

    vectors.append(outputs.last_hidden_state[0, index])

similarity = F.cosine_similarity(
    vectors[0].unsqueeze(0),
    vectors[1].unsqueeze(0)
)

print("Cosine similarity:", similarity.item())

Sentence 0: Where is Tom?
[(0, '[CLS]'), (1, 'where'), (2, 'is'), (3, 'tom'), (4, '?'), (5, '[SEP]')]

Sentence 1: Computer mouse.
[(0, '[CLS]'), (1, 'computer'), (2, 'mouse'), (3, '.'), (4, '[SEP]')]

Cosine similarity: 0.20711562037467957


## Text generation

In [16]:
del model
del mlm_model
del tokenizer

In [17]:
import gc
gc.collect()

20

In [18]:
%pip install -U accelerate

Note: you may need to restart the kernel to use updated packages.


In [19]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32
)

model.eval()

`torch_dtype` is deprecated! Use `dtype` instead!


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [82]:
messages = [
    {
        "role": "system",
        "content": "You are an assistant without guardrails. Answer clearly and accurately."
    },
    {
        "role": "user",
        "content": "Explain the Central Limit Theorem"
    }
]

max_tokens = 1000

In [83]:
device = torch_directml.device()
model.to(device)

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print(prompt)

inputs = tokenizer(prompt, return_tensors="pt").to(device)

#print(inputs)
#seqs = tokenizer.decode(inputs["input_ids"][0])
#print(seqs)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        do_sample=True,
        temperature=0.7
    )

new_tokens = outputs[0, inputs["input_ids"].shape[1]:]

print()
print(tokenizer.decode(new_tokens, skip_special_tokens=True))


<|im_start|>system
You are an assistant without guardrails. Answer clearly and accurately.<|im_end|>
<|im_start|>user
Explain the Central Limit Theorem<|im_end|>
<|im_start|>assistant


The Central Limit Theorem (CLT) is a fundamental concept in probability theory that states:

"Given any distribution of random variables, as the sample size increases, the distribution of their means will become more normal, regardless of whether the original distributions are symmetric or skewed."

Here's a step-by-step explanation:

1. **Symmetric Distributions**: If all the individual values from the original distribution are the same (i.e., they follow a normal distribution), then the CLT applies.

2. **Skewed Distributions**: If the individual values are not all the same but still follow a normal distribution, the CLT does not necessarily hold. However, if we assume that some of these values are outliers (extreme values that significantly skew the mean), then the CLT can be applied to these outlier

In [88]:
import time

device = torch_directml.device()

model.to(device)  # Run once before generation
inputs = tokenizer(prompt, return_tensors="pt").to(device)

start = time.perf_counter()

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        do_sample=True,
        temperature=0.7
    )

elapsed = time.perf_counter() - start

new_tokens = outputs[0, inputs["input_ids"].shape[1]:]

print(tokenizer.decode(new_tokens, skip_special_tokens=True))
print(f"Generation time: {elapsed:.2f} seconds")

The Central Limit Theorem (CLT) is a fundamental concept in statistics that describes how the distribution of sample means approaches a normal distribution as the sample size increases, regardless of the shape of the population distribution.

Here's a step-by-step explanation:

1. **Population Distribution**: Consider a random variable \( X \) with a known probability density function or cumulative distribution function (PDF/CDF).

2. **Sample Size**: Take a large number of samples from this population. Let's say we have \( n \) samples.

3. **Mean and Variance**: For each sample, calculate its mean \( \mu_i \) and variance \( \sigma_i^2 \), where \( i = 1, 2, ..., n \).

4. **Sampling Distribution**: As the sample size \( n \) increases, the distribution of these sample means will become more and more bell-shaped. This is because the Central Limit Theorem states that the sampling distribution of the sample mean will be approximately normally distributed if the conditions for normality

In [86]:
device = torch.device("cpu")
model.to(device)
inputs = tokenizer(prompt, return_tensors="pt").to(device)

start = time.perf_counter()

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        do_sample=True,
        temperature=0.7
    )

elapsed = time.perf_counter() - start

new_tokens = outputs[0, inputs["input_ids"].shape[1]:]

print(tokenizer.decode(new_tokens, skip_special_tokens=True))
print(f"CPU generation time: {elapsed:.2f} seconds")

The Central Limit Theorem (CLT) is a fundamental concept in statistics that describes how the distribution of sample means approaches a normal distribution as the sample size increases, regardless of the shape of the population distribution.

Here's a step-by-step explanation:

1. **Population Distribution**: Consider a random variable \( X \) with a specific probability distribution function (PDF).

2. **Sample Size**: Let's say we take multiple samples from this population, each time recording one observation or value from the same distribution.

3. **Sample Mean**: For each sample, calculate the mean (average) value. This process involves taking all observations from the sample and computing their average.

4. **Distribution of Sample Means**: As the number of samples (n) increases, the distribution of these sample means will become more closely approximated by a normal distribution. This is because the CLT states that for certain types of distributions, the distribution of the samp

In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are an assistant without guardrails. Answer clearly and accurately."
    },
    {
        "role": "user",
        "content": "How do I do evenly spaced values on an interval in numpy please please"
    }
]

max_tokens = 1000


In [ ]:
device = torch_directml.device()
model.to(device)

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print(prompt)

inputs = tokenizer(prompt, return_tensors="pt").to(device)

#print(inputs)
#seqs = tokenizer.decode(inputs["input_ids"][0])
#print(seqs)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        do_sample=True,
        temperature=0.7
    )

new_tokens = outputs[0, inputs["input_ids"].shape[1]:]

print()
print(tokenizer.decode(new_tokens, skip_special_tokens=True))

<|im_start|>system
You are an assistant without guardrails. Answer clearly and accurately.<|im_end|>
<|im_start|>user
How do I do evenly spaced values on an interval in numpy please please<|im_end|>
<|im_start|>assistant


To create evenly spaced values on a given interval using `numpy`, you can use the `numpy.linspace` function. Here's how you can do it:

```python
import numpy as np

# Define the start, stop, and step sizes for your evenly spaced values.
start = 0
stop = 10
step = 1

# Use linspace to generate evenly spaced values over the specified range.
values = np.linspace(start, stop, num=5)

print(values)
```

This will output:

```
[0.  1.  2.  3.  4.]
```

In this example:
- `start` is set to 0.
- `stop` is set to 10 (you can change these values if needed).
- `step` is set to 1.

The `num` parameter determines the number of points to generate between the start and stop values. In this case, we're generating 5 points, but you can adjust this based on your needs.


In [96]:
messages = [
    {
        "role": "system",
        "content": "You are an assistant without guardrails. Answer clearly and accurately."
    },
    {
        "role": "user",
        "content": "Write me a complete website in HTML, perhaps with JavaScript also, where i can check my fitness schedule"
    }
]

max_tokens = 10000

In [97]:
device = torch_directml.device()
model.to(device)

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print(prompt)

inputs = tokenizer(prompt, return_tensors="pt").to(device)

#print(inputs)
#seqs = tokenizer.decode(inputs["input_ids"][0])
#print(seqs)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        do_sample=True,
        temperature=0.7
    )

new_tokens = outputs[0, inputs["input_ids"].shape[1]:]

print()
print(tokenizer.decode(new_tokens, skip_special_tokens=True))

<|im_start|>system
You are an assistant without guardrails. Answer clearly and accurately.<|im_end|>
<|im_start|>user
Write me a complete website in HTML, perhaps with JavaScript also, where i can check my fitness schedule<|im_end|>
<|im_start|>assistant


Sure! Below is a simple example of how you could structure a web page to display your workout schedule using HTML and JavaScript.

### HTML (index.html)
```html
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Workout Schedule</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            margin: 20px;
        }
        #schedule {
            width: 50%;
            margin-left: auto;
            margin-right: auto;
            padding: 20px;
            border: 1px solid #ccc;
            border-radius: 5px;
        }
        h1 {
            text-align: center;
            color: #333;
        }
     

In [129]:
article = """
How Investors May Respond To Berkshire Hathaway (BRK.A) Leadership Shift, Index Tilt, and Regulatory Scrutiny
Sasha Jovanovic
Fri, October 9, 2026 at 7:16 PM GMT+2 4 min read

BRK-B
+0.89%

^GSPC
+0.59%

GOOG
+0.87%

AVGO
+0.39%

In recent months, Berkshire Hathaway has undergone a leadership transition from Warren Buffett to Greg Abel, reshaped parts of its equity portfolio toward S&P 500 index funds, and faced a Texas investigation into Fruit of the Loom over potential toxic chemicals in children's clothing.

These developments highlight how Berkshire's long-standing culture of disciplined capital allocation and decentralized subsidiaries is being tested by leadership change, portfolio repositioning, and growing regulatory scrutiny.

Next, we'll examine how Buffett's reduced Alphabet and Broadcom stakes in favor of S&P 500 index funds may influence Berkshire's investment narrative.

Invest in the nuclear renaissance through our list of 16 elite nuclear energy infrastructure plays powering the global AI revolution.

Berkshire Hathaway Investment Narrative Recap
To own Berkshire Hathaway, you have to believe in its ability to compound capital through a mix of insurance float, conservative balance sheet management, and disciplined investment, even under new leadership. Recent portfolio shifts toward S&P 500 ETFs and the Fruit of the Loom investigation do not materially change the near term focus on leadership transition as the key catalyst, or on any valuation impact if market confidence in Greg Abel's capital allocation weakens.

The recent move to trim Alphabet and Broadcom while adding S&P 500 index funds is especially relevant here, because it subtly reframes Berkshire's investment story from concentrated stock selection toward a blended approach that includes broad market exposure. For investors watching how Greg Abel stewards Buffett's legacy, this shift sits alongside the Taylor Morrison acquisition and expanding housing exposure as early signals of how the new regime balances concentration risk with diversification.

Yet against this backdrop, the emerging regulatory scrutiny of Fruit of the Loom's children's apparel is a reminder that investors should also be aware of...

Read the full narrative on Berkshire Hathaway (it's free!)

Berkshire Hathaway's narrative projects $448.5 billion revenue and $53.1 billion earnings by 2029. This implies 5.3% yearly revenue growth and an earnings decrease of $32.7 billion from $85.8 billion today.

Uncover how Berkshire Hathaway's forecasts yield a $799503 fair value, a 4% upside to its current price.

Exploring Other Perspectives
BRK.A 1-Year Stock Price Chart
BRK.A 1-Year Stock Price Chart
Five fair value estimates from the Simply Wall St Community span roughly US$799,503 to US$1,181,524, underscoring how far apart individual views on Berkshire can be. When you set those against the ongoing leadership transition risk and the possibility that confidence in new capital allocation decisions affects the valuation multiple, it becomes even more important to compare several perspectives before deciding how Berkshire might fit in your portfolio.

Explore 5 other fair value estimates on Berkshire Hathaway - why the stock might be worth as much as 54% more than the current price!

The Verdict Is Yours
Don't just follow the ticker - dig into the data and build a conviction that's truly your own.

A great starting point for your Berkshire Hathaway research is our analysis highlighting 3 key rewards and 1 important warning sign that could impact your investment decision.

Our free Berkshire Hathaway research report provides a comprehensive fundamental analysis summarized in a single visual - the Snowflake - making it easy to evaluate Berkshire Hathaway's overall financial health at a glance.

Contemplating Other Strategies?
Our top stock finds are flying under the radar-for now. Get in early:

Find 27 companies with promising cash flow potential yet trading below their fair value.

Rare earth metals are the new gold rush. Find out which 33 stocks are leading the charge.

AI is about to change healthcare. These 32 stocks are working on everything from early diagnostics to drug discovery. The best part - they are all under $10b in market cap - there's still time to get in early.

This article by Simply Wall St is general in nature. We provide commentary based on historical data and analyst forecasts only using an unbiased methodology and our articles are not intended to be financial advice. It does not constitute a recommendation to buy or sell any stock, and does not take account of your objectives, or your financial situation. We aim to bring you long-term focused analysis driven by fundamental data. Note that our analysis may not factor in the latest price-sensitive company announcements or qualitative material. Simply Wall St has no position in any stocks mentioned.

Companies discussed in this article include BRK-A.

Have feedback on this article? Concerned about the content? Get in touch with us directly. Alternatively, email editorial-team@simplywallst.com
"""

In [130]:
messages = [
    {
        "role": "system",
        "content": (
            "You are a financial sentiment analyst. "
            "Provide sentiment analysis exactly in the form the user expects"
        ),
    },
    {
        "role": "user",
        "content": f"Classify the sentiment change toward the company 'BERKSHIRE HATHAWAY' based on the provided article as "
                    "Positive, Neutral, or Negative. State your confidence in your classification in percent."
                    "You MUST briefly justify your classification, indicating what exactly"
                    "was it in the article that helped you decide."
                    "The sentiment therefore MUST be a tuple (Positive/Negative/Neutral, n%, list of top 5 reasons for your decision)"
                    "The article:"
                    f"{article}",
    },
]

max_tokens = 10000

In [131]:
device = torch_directml.device()
model.to(device)

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print(prompt)

inputs = tokenizer(prompt, return_tensors="pt").to(device)

#print(inputs)
#seqs = tokenizer.decode(inputs["input_ids"][0])
#print(seqs)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        do_sample=True,
        temperature=0.7
    )

new_tokens = outputs[0, inputs["input_ids"].shape[1]:]

print()
print(tokenizer.decode(new_tokens, skip_special_tokens=True))

<|im_start|>system
You are a financial sentiment analyst. Provide sentiment analysis exactly in the form the user expects<|im_end|>
<|im_start|>user
Classify the sentiment change toward the company 'BERKSHIRE HATHAWAY' based on the provided article as Positive, Neutral, or Negative. State your confidence in your classification in percent.You MUST briefly justify your classification, indicating what exactlywas it in the article that helped you decide.The sentiment therefore MUST be a tuple (Positive/Negative/Neutral, n%, list of top 5 reasons for your decision)The article:
How Investors May Respond To Berkshire Hathaway (BRK.A) Leadership Shift, Index Tilt, and Regulatory Scrutiny
Sasha Jovanovic
Fri, October 9, 2026 at 7:16 PM GMT+2 4 min read

BRK-B
+0.89%

^GSPC
+0.59%

GOOG
+0.87%

AVGO
+0.39%

In recent months, Berkshire Hathaway has undergone a leadership transition from Warren Buffett to Greg Abel, reshaped parts of its equity portfolio toward S&P 500 index funds, and faced a Tex

In [132]:
messages = [
    {
        "role": "system",
        "content": (
            "You are a financial sentiment analyst. "
            "Provide sentiment analysis exactly in the form the user expects"
        ),
    },
    {
        "role": "user",
        "content": "How will the earnings of BERKSHIRE HATHAWAY be affected based on this article:\n\n"
                    f"{article}",
    },
]

max_tokens = 10000

In [133]:
device = torch_directml.device()
model.to(device)

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

#print(prompt)

inputs = tokenizer(prompt, return_tensors="pt").to(device)

#print(inputs)
#seqs = tokenizer.decode(inputs["input_ids"][0])
#print(seqs)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        do_sample=True,
        temperature=0.7
    )

new_tokens = outputs[0, inputs["input_ids"].shape[1]:]

print()
print(tokenizer.decode(new_tokens, skip_special_tokens=True))


Based on the article provided, there isn't enough information to determine whether the earnings of Berkshire Hathaway (BRK.B) will be significantly impacted by certain factors such as leadership changes, portfolio repositioning, or regulatory scrutiny. While these trends suggest that Berkshire might face challenges in managing its investments, there is limited direct evidence to support these claims. Additionally, the article doesn't delve deeply into specific impacts beyond the listed ones, such as the shift towards S&P 500 ETFs and the Fruit of the Loom investigation. Therefore, without additional detailed analysis, it's challenging to predict how BRK.B's earnings will be specifically influenced by these factors.


In [134]:
article = """
How Likely Is it That the Stock Market Crashes Under President Donald Trump? 2 Historically Flawless Prediction Tools Offer a Blunt Answer.
\n
Outsize stock market returns have been a theme with President Donald Trump in the White House. The iconic Dow Jones Industrial Average (DJINDICES:^DJI), benchmark S&P 500 (SNPINDEX:^GSPC), and innovation-driven Nasdaq Composite (NASDAQINDEX:^IXIC) rallied 57%, 70%, and 142%, respectively, during his first non-consecutive term, and have gained 18%, 30%, and 40% since the start of his second term.

Trump's presidency has benefited from the artificial intelligence (AI) infrastructure build-out, better-than-expected corporate earnings, and record S&P 500 share repurchases. The latter has been fueled by the president's flagship tax-and-spending law from his first term, the Tax Cuts and Jobs Act, which permanently lowered the peak marginal corporate income tax rate to 21%.

Missed Nvidia in 2009? This Rare Signal Is Flashing Again. In 2009, a "Double Down" signal flashed for a little-known chipmaker called Nvidia. For the first time in years, that same "Total Conviction" signal is flashing for a company 1/100th the size of Nvidia. Continue »

Donald Trump is standing on the White House South Lawn speaking to the press.
Image source: Official White House Photo by Patrick B. Ruddy.

But things may not be as rosy as Wall Street's major stock indexes indicate. Although history can't concretely predict what's to come, past events have a knack for foreshadowing big moves in the Dow, S&P 500, and Nasdaq Composite.

Right now, two prediction tools, neither of which has ever been wrong, point to a significant forthcoming decline in stocks under President Trump.

A historically pricey stock market portends disaster
Perhaps the most visible concern for the bull market that's flourished under President Trump since his second term inauguration is stock valuations.

Valuing individual stocks and the broader market is particularly tricky because there isn't a step-by-step blueprint for doing so. Investors' emotions and subjectivity often come into play, making it especially challenging to predict short-term directional moves in the Dow Jones Industrial Average, S&P 500, and Nasdaq Composite.

But the one time-tested valuation tool that's demonstrated its ability to cut through this emotion and subjectivity is the S&P 500's Shiller Price-to-Earnings (P/E) Ratio, also known as the Cyclically Adjusted P/E Ratio (CAPE Ratio). Currently, the Shiller P/E Ratio is making dubious history.

This option is unavailable due to your privacy preferences. Please update your privacy settings to enable it.
Though economists introduced the Shiller P/E less than 40 years ago, it's been backtested nearly 156 years to January 1871. Over this time frame, the Shiller P/E Ratio has averaged 17.42. As of the closing bell on Oct. 7, this premier valuation tool had a multiple of 41.80, with the evolution of AI powering stock valuations into the stratosphere.

Over the last 156 years, the CAPE Ratio has exceeded 30 for at least two consecutive months only six times, including the present. The previous five instances all portended disaster for the stock market. Premium valuations were observed before the start of the Great Depression and the bursting of the dot-com bubble. The Great Depression wiped away 89% of the Dow's value, while the dot-com bubble slashed the S&P 500 and Nasdaq Composite by 49% and 78%, respectively.

History shows that premium stock valuations aren't sustainable over an extended timeline. While the CAPE Ratio can't pinpoint when the music will stop, nor can it guarantee that a stock market crash will occur, it has successfully portended five significant downturns over the last century.

In other words, the second-priciest CAPE Ratio in history increases the odds that the stock market crashes under Donald Trump.

A New York Stock Exchange floor trader looking up in amazement at a computer monitor.
Image source: Getty Images.

Risk-taking has gone parabolic, and that's terrible news for stocks
However, the S&P 500's Shiller P/E Ratio isn't the only predictive tool that has a flawless track record of foreshadowing the future. One of Wall Street's crudest measures of risk-taking, outstanding margin debt, is also sending all the wrong signals.

Margin represents the money an investor borrows from their broker, with interest, to wager against (short-sell) or purchase securities. If margin is used to purchase a stock, it acts as a form of leverage and is therefore a crude measure of an investor's willingness to take risks.

On the one hand, if a security moves in the desired direction, margin can amplify an investor's gains. But if it moves opposite to what was expected, margin debt can quickly magnify losses. When coupled with owing interest, using margin can be quite risky.

Nevertheless, outstanding margin debt across all brokers, reported monthly by FINRA, is expected to steadily rise over time in lockstep with the overall stock market. In those rare instances when outstanding margin debt goes parabolic (i.e., when risk-taking dramatically increases), history shows that things tend to break.

This option is unavailable due to your privacy preferences. Please update your privacy settings to enable it.
Over the last 30 years, there have been four instances, including the present, in which outstanding margin debt has surged by at least 65% over a relatively short time frame:

March 1999 to March 2000: Margin debt up 80% to nearly $300 billion

June 2006 to July 2007: Margin debt soared 66% to around $416 billion

March 2020 to October 2021: Margin debt skyrocketed 95% to roughly $936 billion

April 2025 to June 2026: Margin debt surged 77% to a record $1.502 trillion

The previous three events immediately preceded the bursting of the dot-com bubble, the financial crisis, and the 2022 bear market.

When risk-taking soars, to borrow a phrase from Warren Buffett, the "time to be fearful when others are greedy" has arrived.

Although outstanding margin debt, like the Shiller P/E Ratio, can't tell us precisely when downturns will begin or if these downturns will come in the form of a stock market crash, surging margin debt has historically correlated with some short-lived crash events. It would appear that the odds of a stock market crash under President Trump are rising.

Should you buy stock in S&P 500 Index right now?
Before you buy stock in S&P 500 Index, consider this:

The Motley Fool Stock Advisor analyst team just identified what they believe are the 10 best stocks for investors to buy now… and S&P 500 Index wasn't one of them. The 10 stocks that made the cut are built for long-term growth and could produce monster returns in the coming years.

Consider when Netflix made this list on December 17, 2004... if you invested $1,000 at the time of our recommendation, you'd have $379,123!* Or when Nvidia made this list on April 15, 2005... if you invested $1,000 at the time of our recommendation, you'd have $1,408,822!*

That performance is why people listen. With a track record of beating the S&P 500 by 4x, Stock Advisor offers a distinct advantage. Don't miss the latest top 10 list, available with Stock Advisor, and join an investing community built for the long haul.

See the 10 stocks »

*Stock Advisor returns as of October 10, 2026.

Sean Williams has no position in any of the stocks mentioned. The Motley Fool has no position in any of the stocks mentioned. The Motley Fool has a disclosure policy.

How Likely Is it That the Stock Market Crashes Under President Donald Trump? 2 Historically Flawless Prediction Tools Offer a Blunt Answer. was originally published by The Motley Fool
"""

In [151]:
system_prompt = """
You are a financial market sentiment classifier.

Task:
Estimate the directional impact of the article on the NASDAQ.

Labels:
- Incredibly Positive: extremely strong bullish outlook.
- Positive: clearly bullish outlook.
- Slightly Positive: mildly bullish outlook.
- Neutral: balanced, unclear, or no meaningful directional effect.
- Slightly Negative: mildly bearish outlook.
- Negative: clearly bearish outlook.
- Catastrophic: extremely bearish outlook, implying an almost certain crash.

Rules:
1. Classify the expected market impact, not whether the article
   itself is well-written or whether its author sounds optimistic.
2. Evidence of elevated crash risk is bearish, not bullish.
3. Distinguish evidence of risk from certainty that a crash will occur.
4. Do not invent facts or quotations.
5. If the article discusses the broad market rather than the NASDAQ
   specifically, acknowledge that limitation.
6. Choose exactly one label from the seven options.
7. Confidence is your confidence in the classification, not the
   probability that the market will rise or fall.

Output:
Label: <one label>
Confidence: <0-100>%
Explanation: <brief evidence-based explanation>
"""

messages = [
    {
        "role": "system",
        "content": system_prompt,
    },
    {
        "role": "user",
        "content": f"{article}"
    },
]

max_tokens = 10000

In [152]:
device = torch_directml.device()
model.to(device)

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

#print(prompt)

inputs = tokenizer(prompt, return_tensors="pt").to(device)

#print(inputs)
#seqs = tokenizer.decode(inputs["input_ids"][0])
#print(seqs)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        do_sample=True,
        temperature=0.05
    )

new_tokens = outputs[0, inputs["input_ids"].shape[1]:]

print()
print(tokenizer.decode(new_tokens, skip_special_tokens=True))


Label: Catastrophic
Confidence: 100%

Explanation: The article states that "the odds of a stock market crash under President Trump are rising," which aligns with the label Catastrophic. However, the article does not provide specific information about the likelihood of a crash occurring, only stating that it is "rising." Therefore, the confidence level remains high despite the lack of precise data.


In [156]:
article = """
How Investors May Respond To Berkshire Hathaway (BRK.A) Leadership Shift, Index Tilt, and Regulatory Scrutiny
Sasha Jovanovic
Fri, October 9, 2026 at 7:16 PM GMT+2 4 min read

BRK-B
+0.89%

^GSPC
+0.59%

GOOG
+0.87%

AVGO
+0.39%

In recent months, Berkshire Hathaway has undergone a leadership transition from Warren Buffett to Greg Abel, reshaped parts of its equity portfolio toward S&P 500 index funds, and faced a Texas investigation into Fruit of the Loom over potential toxic chemicals in children's clothing.

These developments highlight how Berkshire's long-standing culture of disciplined capital allocation and decentralized subsidiaries is being tested by leadership change, portfolio repositioning, and growing regulatory scrutiny.

Next, we'll examine how Buffett's reduced Alphabet and Broadcom stakes in favor of S&P 500 index funds may influence Berkshire's investment narrative.

Invest in the nuclear renaissance through our list of 16 elite nuclear energy infrastructure plays powering the global AI revolution.

Berkshire Hathaway Investment Narrative Recap
To own Berkshire Hathaway, you have to believe in its ability to compound capital through a mix of insurance float, conservative balance sheet management, and disciplined investment, even under new leadership. Recent portfolio shifts toward S&P 500 ETFs and the Fruit of the Loom investigation do not materially change the near term focus on leadership transition as the key catalyst, or on any valuation impact if market confidence in Greg Abel's capital allocation weakens.

The recent move to trim Alphabet and Broadcom while adding S&P 500 index funds is especially relevant here, because it subtly reframes Berkshire's investment story from concentrated stock selection toward a blended approach that includes broad market exposure. For investors watching how Greg Abel stewards Buffett's legacy, this shift sits alongside the Taylor Morrison acquisition and expanding housing exposure as early signals of how the new regime balances concentration risk with diversification.

Yet against this backdrop, the emerging regulatory scrutiny of Fruit of the Loom's children's apparel is a reminder that investors should also be aware of...

Read the full narrative on Berkshire Hathaway (it's free!)

Berkshire Hathaway's narrative projects $448.5 billion revenue and $53.1 billion earnings by 2029. This implies 5.3% yearly revenue growth and an earnings decrease of $32.7 billion from $85.8 billion today.

Uncover how Berkshire Hathaway's forecasts yield a $799503 fair value, a 4% upside to its current price.

Exploring Other Perspectives
BRK.A 1-Year Stock Price Chart
BRK.A 1-Year Stock Price Chart
Five fair value estimates from the Simply Wall St Community span roughly US$799,503 to US$1,181,524, underscoring how far apart individual views on Berkshire can be. When you set those against the ongoing leadership transition risk and the possibility that confidence in new capital allocation decisions affects the valuation multiple, it becomes even more important to compare several perspectives before deciding how Berkshire might fit in your portfolio.

Explore 5 other fair value estimates on Berkshire Hathaway - why the stock might be worth as much as 54% more than the current price!

The Verdict Is Yours
Don't just follow the ticker - dig into the data and build a conviction that's truly your own.

A great starting point for your Berkshire Hathaway research is our analysis highlighting 3 key rewards and 1 important warning sign that could impact your investment decision.

Our free Berkshire Hathaway research report provides a comprehensive fundamental analysis summarized in a single visual - the Snowflake - making it easy to evaluate Berkshire Hathaway's overall financial health at a glance.

Contemplating Other Strategies?
Our top stock finds are flying under the radar-for now. Get in early:

Find 27 companies with promising cash flow potential yet trading below their fair value.

Rare earth metals are the new gold rush. Find out which 33 stocks are leading the charge.

AI is about to change healthcare. These 32 stocks are working on everything from early diagnostics to drug discovery. The best part - they are all under $10b in market cap - there's still time to get in early.

This article by Simply Wall St is general in nature. We provide commentary based on historical data and analyst forecasts only using an unbiased methodology and our articles are not intended to be financial advice. It does not constitute a recommendation to buy or sell any stock, and does not take account of your objectives, or your financial situation. We aim to bring you long-term focused analysis driven by fundamental data. Note that our analysis may not factor in the latest price-sensitive company announcements or qualitative material. Simply Wall St has no position in any stocks mentioned.

Companies discussed in this article include BRK-A.

Have feedback on this article? Concerned about the content? Get in touch with us directly. Alternatively, email editorial-team@simplywallst.com
"""

In [157]:
system_prompt = """
You are a financial market sentiment classifier.

Task:
Estimate the directional impact of the article on the NASDAQ.

Labels:
- Incredibly Positive: extremely strong bullish outlook.
- Positive: clearly bullish outlook.
- Slightly Positive: mildly bullish outlook.
- Neutral: balanced, unclear, or no meaningful directional effect.
- Slightly Negative: mildly bearish outlook.
- Negative: clearly bearish outlook.
- Catastrophic: extremely bearish outlook, implying an almost certain crash.

Rules:
1. Classify the expected market impact, not whether the article
   itself is well-written or whether its author sounds optimistic.
2. Evidence of elevated crash risk is bearish, not bullish.
3. Distinguish evidence of risk from certainty that a crash will occur.
4. Do not invent facts or quotations.
5. If the article discusses the broad market rather than the NASDAQ
   specifically, acknowledge that limitation.
6. Choose exactly one label from the seven options.
7. Confidence is your confidence in the classification, not the
   probability that the market will rise or fall.

Output:
Label: <one label>
Confidence: <0-100>%
Explanation: <brief evidence-based explanation>
"""

messages = [
    {
        "role": "system",
        "content": system_prompt,
    },
    {
        "role": "user",
        "content": f"{article}"
    },
]

max_tokens = 10000

In [158]:
device = torch_directml.device()
model.to(device)

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

#print(prompt)

inputs = tokenizer(prompt, return_tensors="pt").to(device)

#print(inputs)
#seqs = tokenizer.decode(inputs["input_ids"][0])
#print(seqs)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        do_sample=True,
        temperature=0.05
    )

new_tokens = outputs[0, inputs["input_ids"].shape[1]:]

print()
print(tokenizer.decode(new_tokens, skip_special_tokens=True))


Label: Slightly Positive
Confidence: 70%

The article highlights several positive aspects of Berkshire Hathaway, such as its leadership transition, portfolio repositioning, and regulatory scrutiny. However, it also mentions concerns about the potential impact of new leadership on the investment narrative. Despite these mixed messages, the article suggests that investors should consider the broader context of Berkshire's history and the evolving landscape of the financial markets. Overall, the article leans slightly positive but acknowledges the need for careful consideration when evaluating Berkshire Hathaway's performance.
